# 主動型投資- 資料處理
**以「週」為單位做台股的買賣，賺價差**
流程 (網路爬蟲都使用 `WeekTermInvestment_DataScrape` 的工具)：
1. 輸入台股代碼 (ABBR)
2. 檢查 DataCollection/ABBR 資料夾內的 5 份 CSV (法人買賣超、資券餘額、借券賣出餘額、大戶籌碼、股價週K線)
3. 沒有資料夾或檔案有誤時：先用 `WeekTermInvestment_DataScrape` 抓取新資料，抓取成功後再刪除 DataCollection/ABBR 資料夾內的舊資料 (若有的話)，再存入新資料
4. 設定分析用的週次變數

投資操作：
* 買_時機點：大戶籌碼比例增加 & 大戶借券量下降 & 融資(韭菜)餘額下降，買進價格<=大戶平均成本
* 賣_時機點：大戶籌碼比例下降 & 大戶借券量增加 & 融資(韭菜)餘額上升 & 融資維持率持平/下降，賣出價格>=韭菜平均成本

### 匯入Package

In [2]:
import ast
import builtins
import json
import re
import types
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = Path.cwd()  # 本 notebook 所在的資料夾
DATASCRAPE_NOTEBOOK = BASE_DIR / "WeekTermInvestment_DataScrape.ipynb"


def _is_definition(node, namespace):
    """只保留 import、函式/類別定義、常數設定。
    常數設定的等號右邊不能呼叫函式 (以免執行到 input() 或爬蟲)，用到的變數也必須已經定義
    (例如執行格中的 outputs = [(df_inst, ...)] 會被略過，因為 df_inst 是爬蟲的結果)"""
    if isinstance(node, (ast.Import, ast.ImportFrom, ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
        return True
    if isinstance(node, (ast.Assign, ast.AnnAssign)) and node.value is not None:
        sub_nodes = list(ast.walk(node.value))
        names = [n.id for n in sub_nodes if isinstance(n, ast.Name)]
        return not any(isinstance(n, ast.Call) for n in sub_nodes) and all(
            name in namespace or hasattr(builtins, name) for name in names
        )
    return False


def load_datascrape_tools(nb_path):
    """把 WeekTermInvestment_DataScrape.ipynb 當成 package 載入：
    只執行裡面的 import 與函式/常數定義，不會執行「輸入代碼 → 爬資料」那一格"""
    nb = json.loads(Path(nb_path).read_text(encoding="utf-8"))
    module = types.ModuleType(Path(nb_path).stem)
    module.__file__ = str(nb_path)
    for cell in nb["cells"]:
        if cell["cell_type"] != "code":
            continue
        try:
            tree = ast.parse("".join(cell["source"]))
        except SyntaxError:  # 例如含有 %magic 的格子，不是工具定義，略過
            continue
        # 逐行判斷、逐行執行，才知道某個變數在這一行之前是否已經定義
        for node in tree.body:
            if _is_definition(node, module.__dict__):
                exec(compile(ast.Module(body=[node], type_ignores=[]), str(nb_path), "exec"), module.__dict__)
    return module


# 所有網路爬蟲 (籌碼資料、週K線股價) 都在 DataScrape 中
ds = load_datascrape_tools(DATASCRAPE_NOTEBOOK)
DATA_DIR = BASE_DIR / ds.DATA_DIR  # 各股票的資料夾 (ABBR) 都放在這裡 (DataCollection，定義在 DataScrape)
print(f"已載入 {DATASCRAPE_NOTEBOOK.name} 的工具")

已載入 WeekTermInvestment_DataScrape.ipynb 的工具


### 函式定義(週資料檔案檢查)

In [1]:
# ===== 週資料：檢查 ABBR 資料夾內的 5 份 CSV，沒有、有誤或太舊就用 DataScrape 重新取得 =====
# 5 份資料的檔名標籤與欄位定義在 DataScrape 的 WEEK_FILE_COLUMNS
# (法人買賣超、資券餘額、借券賣出餘額、大戶籌碼、股價週K線)
# 週次一律用 ISO 年週 (年, 第幾週)；抓資料時還沒結束的那一週資料不完整，不算完整週 (見 _first_unfinished_week)


def _is_yyyymmdd(s):
    try:
        datetime.strptime(s, "%Y%m%d")
        return True
    except ValueError:
        return False


def _latest_week_file(folder, abbr, file_tag):
    """回傳資料夾內 ABBR_{file_tag}_YYYYMMDD.csv 中日期最新的那份，沒有就回傳 None"""
    pattern = re.compile(rf"{abbr}_{file_tag}_(\d{{8}})\.csv")
    matches = [f for f in Path(folder).glob("*.csv") if (m := pattern.fullmatch(f.name)) and _is_yyyymmdd(m.group(1))]
    return max(matches, key=lambda f: f.name, default=None)


def _first_unfinished_week(csv_path):
    """依檔名的抓取日期 (結尾的 YYYYMMDD) 回傳第一個資料不完整的年週：
    週一 ~ 週五抓的是當週 (還沒收盤，或法人等資料還沒公布)；週六、週日抓的 (當週已收盤) 是下一週"""
    created = datetime.strptime(Path(csv_path).stem[-8:], "%Y%m%d")
    if created.weekday() >= 5:
        created += timedelta(days=7)
    return created.isocalendar()[:2]


def _week_monday(year_week):
    """ISO 年週那一週的週一 (date)"""
    return datetime.fromisocalendar(year_week[0], year_week[1], 1).date()


def check_week_files(folder, abbr):
    """檢查資料夾內的 5 份週資料 CSV，回傳問題清單 (空清單代表檔案都正確)
    檢查項目：至少 5 份 CSV、5 種檔名都有 (結尾是合法的年月日)、檔案讀得到且欄位數正確、有資料"""
    n_csv = len(list(Path(folder).glob("*.csv")))
    problems = []
    if n_csv < len(ds.WEEK_FILE_COLUMNS):
        problems.append(f"CSV 檔只有 {n_csv} 份，至少要 {len(ds.WEEK_FILE_COLUMNS)} 份")
    for tag, columns in ds.WEEK_FILE_COLUMNS.items():
        latest = _latest_week_file(folder, abbr, tag)  # 同一種資料有多份時，檢查日期最新的那份
        if latest is None:
            problems.append(f"缺少 {abbr}_{tag}_檔案建立年月日.csv")
            continue
        try:
            df = pd.read_csv(latest, encoding="utf-8-sig")
        except Exception as e:
            problems.append(f"{latest.name} 無法讀取：{e}")
            continue
        if df.shape[1] != len(columns):
            problems.append(f"{latest.name} 有 {df.shape[1]} 欄，應該是 {len(columns)} 欄")
        elif df.empty:
            problems.append(f"{latest.name} 沒有資料")
    return problems


def latest_complete_week(folder, abbr):
    """5 份 CSV 都有資料的最新完整週 (ISO 年, 第幾週)，沒有就回傳 None；只讀取日期欄，不會修改檔案
    每種資料看日期最新的那份 CSV，抓資料時還沒結束的週不算 (見 _first_unfinished_week)"""
    common = None
    for tag in ds.WEEK_FILE_COLUMNS:
        path = _latest_week_file(folder, abbr, tag)
        unfinished = _first_unfinished_week(path)
        first_col = pd.read_csv(path, encoding="utf-8-sig", usecols=[0]).iloc[:, 0]  # 第一欄是日期
        dates = pd.to_datetime(first_col, format="%Y/%m/%d", errors="coerce").dropna()
        weeks = {week for week in (d.isocalendar()[:2] for d in dates) if week < unfinished}
        common = weeks if common is None else common & weeks
    return max(common, default=None)


def _data_age(folder, abbr, this_week, max_lag_weeks):
    """(說明, 是否太舊)：最新的完整週距離 this_week 超過 max_lag_weeks 週就算太舊，沒有完整週也算太舊"""
    latest = latest_complete_week(folder, abbr)
    if latest is None:
        return "5 份 CSV 沒有共同的完整週", True
    lag = (_week_monday(this_week) - _week_monday(latest)).days // 7
    too_old = lag > max_lag_weeks
    text = f"最新的完整週是 {latest[0]}W{latest[1]:02d}，今天在 {this_week[0]}W{this_week[1]:02d}，相差 {lag} 週"
    return text + (f" (>= {max_lag_weeks + 1} 週)" if too_old else ""), too_old


def ensure_week_data(abbr, data_dir=None, max_lag_weeks=1, today=None):
    """確保 ABBR 資料夾內有正確、而且不會太舊的 5 份週資料 CSV，回傳資料夾路徑
    以下情況會用 DataScrape 重新抓取 5 份資料：
      1. 沒有資料夾，或檔案有誤 (見 check_week_files)
      2. 資料太舊：最新的完整週 (見 latest_complete_week) 距離今天所在的週超過 max_lag_weeks 週
         (預設 1 週，也就是相差 >= 2 週；例如今天在第 48 週，最新的完整週只到第 46 週)
    重新抓取時先抓 5 份新資料，全部成功後才刪除資料夾內舊的 CSV (若有的話)，再存入新資料；
    抓取失敗時舊檔都不會被刪除：資料太舊的情況印出提醒並繼續使用舊資料，其他情況直接報錯"""
    data_dir = Path(data_dir or DATA_DIR)
    this_week = (today or datetime.now().date()).isocalendar()[:2]
    folder = data_dir / abbr
    stale = False  # True：檔案都正確、只是太舊 (抓取失敗時還可以繼續用舊資料)
    if not folder.is_dir():
        print(f"⚠ 找不到資料夾「{abbr}」")
    else:
        problems = check_week_files(folder, abbr)
        if problems:
            print(f"⚠ 資料夾「{abbr}」內的檔案有誤：")
            for p in problems:
                print(f"   - {p}")
        else:
            age, stale = _data_age(folder, abbr, this_week, max_lag_weeks)
            if not stale:
                print(f"✔ 資料夾「{abbr}」內的 {len(ds.WEEK_FILE_COLUMNS)} 份 CSV 檔都正確，{age}")
                return folder
            print(f"⚠ 資料夾「{abbr}」的資料太舊：{age}")

    print("使用 WeekTermInvestment_DataScrape 抓取新資料，抓取成功後再刪除舊的 CSV 檔…")
    try:
        week_data = ds.fetch_all_week_data(abbr)
    except Exception as e:
        if not stale:  # 沒有可以繼續用的舊資料
            raise
        print(f"⚠ 抓取失敗，舊的 CSV 檔都保留，繼續使用舊資料 ({type(e).__name__}: {e})")
        return folder
    for old in sorted(folder.glob("*.csv")):  # 資料夾不存在時沒有舊檔
        old.unlink()
        print(f"已刪除舊檔：{old.name}")
    for file_tag, df in week_data.items():
        csv_path = ds.save_week_csv(df, abbr, file_tag, base_dir=data_dir)
        print(f"{df.attrs['symbol']} {file_tag} 共 {len(df)} 筆，已存檔：{csv_path}")

    problems = check_week_files(folder, abbr)
    if problems:
        raise RuntimeError(f"重新取得資料後，資料夾「{abbr}」內的檔案仍有誤：\n" + "\n".join(problems))
    age, stale = _data_age(folder, abbr, this_week, max_lag_weeks)
    print(f"✔ 資料夾「{abbr}」內的 {len(ds.WEEK_FILE_COLUMNS)} 份 CSV 檔都正確，{age}")
    if stale:
        print("⚠ 重新抓取後資料還是太舊：可能這檔股票剛上市、停止交易 (停牌、下市)，或最近有整週休市 (例如春節)")
    return folder

### 函式定義(大戶買入平均成本)
以 `PIVOT_YEAR_WEEK` 為錨點(包含)，往前 `INPUT_WEEK_NUM` 筆法人買賣超週資料，外資、投信+自營商 各自計算期間內的買進均價：
* 賣超不影響成本，只用買超的週：平均成本 = Σ(買超張數 × 當週典型價) / Σ(買超張數)；期間內都沒有買超時為 NaN
* 當週典型價 = (高 + 低 + 收) / 3，來自 `ABBR_Stock_Price_Week`，與法人資料用 ISO 年週對齊
* 投信、自營商視為同一個大戶，同一週的買賣超先相加，再判斷是買超或賣超
* 期初持股與成本無法得知，所以這是大戶「期間內新買進部位」的平均成本

In [3]:
# ===== 平均成本：共用函式 + 大戶（外資、投信+自營商）買入平均成本 =====
# 週次一律用 ISO 年週 (年, 第幾週)：週一到週日為一週，與 date.isocalendar() 相同，
# 例如 date(2026, 11, 6).isocalendar()[:2] == (2026, 45)
# 各份 CSV 同一週的日期不一定相同 (例如法人用週五 11/6、資券餘額用週一 11/2)，所以用 ISO 年週對齊
# 抓資料時還沒結束的那一週資料不完整 (例如週三抓的股價、法人只到週二或週三)，讀取時就排除 (不會修改 csv 檔)

# 大戶分組: 法人買賣超 CSV 中要加總的欄位 (投信+自營商 視為同一個大戶，同一週先相加)
INSTITUTION_GROUPS = {"外資": ["外資(張)"], "投信+自營商": ["投信(張)", "自營商(張)"]}
INSTITUTIONAL_FILE_TAG = "Institutional_Trading_Week_8cols"


def _to_number(col):
    """CSV 中的數字可能是有千分位逗號或百分比的字串 (例如 -2,927、84.17%)，轉成數值 (百分比只去掉 % 符號)；
    無法轉換的 (例如「-」) 變成 NaN；已經是數值的欄位直接回傳"""
    if pd.api.types.is_numeric_dtype(col):
        return col
    return pd.to_numeric(col.astype(str).str.replace(",", "").str.rstrip("%"), errors="coerce")


def read_week_csv(abbr, file_tag, data_dir=None, complete_weeks_only=True):
    """讀取 data_dir/ABBR 資料夾中日期最新的 ABBR_{file_tag}_YYYYMMDD.csv (data_dir 預設為 DATA_DIR)，
    依日期由新到舊排序，並加上「年週」欄位 (ISO 年, 第幾週)
    complete_weeks_only=True (預設)：抓資料時還沒結束的週 (見 _first_unfinished_week) 不列入；
    只是讀進來時排除，不會修改 csv 檔"""
    path = _latest_week_file(Path(data_dir or DATA_DIR) / abbr, abbr, file_tag)
    if path is None:
        raise FileNotFoundError(f"資料夾「{abbr}」中找不到 {abbr}_{file_tag}_檔案建立年月日.csv")
    df = pd.read_csv(path, encoding="utf-8-sig")
    df["_date"] = pd.to_datetime(df.iloc[:, 0], format="%Y/%m/%d")  # 第一欄是日期
    df = df.sort_values("_date", ascending=False).reset_index(drop=True)
    df["年週"] = [d.isocalendar()[:2] for d in df.pop("_date")]
    if complete_weeks_only:
        unfinished = _first_unfinished_week(path)
        df = df.loc[[week < unfinished for week in df["年週"]]].reset_index(drop=True)
    return df


def _week_window(df, pivot_year_week, input_week_num, label):
    """從 read_week_csv 讀到的週資料 (由新到舊) 中，取出以 pivot_year_week 為終點 (包含) 往前 input_week_num 筆"""
    pivot_year_week = tuple(pivot_year_week)
    positions = [i for i, yw in enumerate(df["年週"]) if yw == pivot_year_week]
    if not positions:
        raise ValueError(
            f"{label}資料中沒有 {pivot_year_week} 這一週 (資料範圍 {df['年週'].iloc[-1]} ~ {df['年週'].iloc[0]})"
        )
    window = df.iloc[positions[0] : positions[0] + input_week_num]
    if len(window) < input_week_num:
        raise ValueError(f"{pivot_year_week} 往前只有 {len(window)} 筆{label}資料，不足 {input_week_num} 筆")
    return window


def _week_typical_price(price, window):
    """回傳 window 中每一週的典型價 (高 + 低 + 收) / 3 (比只用收盤價更接近整週的成交價位)，index 與 window 相同
    price 是 read_week_csv 讀到的股價週資料"""
    typical = (_to_number(price["高"]) + _to_number(price["低"]) + _to_number(price["收"])) / 3
    price_by_week = dict(zip(price["年週"], typical))
    missing = [yw for yw in window["年週"] if pd.isna(price_by_week.get(yw))]
    if missing:
        raise ValueError(f"股價資料中沒有這些週的高/低/收：{missing}")
    return pd.Series([price_by_week[yw] for yw in window["年週"]], index=window.index)


def _buy_avg_cost(window, lots, week_price, label):
    """賣出不影響成本，只用買進 (張數 > 0) 的週：平均成本 = Σ(買進張數 × 當週典型價) / Σ(買進張數)
    期初持股與成本無法得知，所以這是「期間內新買進部位」的平均成本；期間內都沒有買進時是 NaN"""
    if lots.isna().any():
        raise ValueError(f"{label} 的張數有無法轉成數字的值，年週：{window.loc[lots.isna(), '年週'].tolist()}")
    buys = lots > 0
    total_lots = lots[buys].sum()
    total_money = (lots[buys] * week_price[buys]).sum()  # 張 × 元；1 張 = 1000 股，相除時單位抵銷，即每股價格
    return total_money / total_lots if total_lots > 0 else float("nan")


def _institutional_avg_cost(inst, price, pivot_year_week, input_week_num):
    """用已讀進來的法人買賣超、股價週資料計算大戶平均成本 (calc_institutional_avg_cost 的計算本體)"""
    window = _week_window(inst, pivot_year_week, input_week_num, "法人買賣超")
    week_price = _week_typical_price(price, window)
    return {
        name: _buy_avg_cost(window, sum(_to_number(window[c]) for c in columns), week_price, name)
        for name, columns in INSTITUTION_GROUPS.items()
    }


def calc_institutional_avg_cost(abbr, pivot_year_week, input_week_num, data_dir=None):
    """計算大戶 (外資、投信+自營商) 在期間內的買進均價，回傳 {"外資": 平均成本, "投信+自營商": 平均成本}
    - 以 pivot_year_week (ISO 年, 第幾週) 為終點 (包含)，往前取 input_week_num 筆法人買賣超週資料
    - 只用買超的週：平均成本 = Σ(買超張數 × 當週典型價) / Σ(買超張數)；期間內都沒有買超時是 NaN"""
    inst = read_week_csv(abbr, INSTITUTIONAL_FILE_TAG, data_dir)
    price = read_week_csv(abbr, ds.PRICE_FILE_TAG, data_dir)
    return _institutional_avg_cost(inst, price, pivot_year_week, input_week_num)

### 函式定義(散戶買入平均成本)
以 `PIVOT_YEAR_WEEK` 為錨點(包含)，往前 `INPUT_WEEK_NUM` 筆資券餘額週資料，算法比照大戶：
* 融資減少不影響成本，只用融資增加的週：平均成本 = Σ(融資增加張數 × 當週典型價) / Σ(融資增加張數)；期間內融資都沒有增加時為 NaN
* 融資增減取自 `ABBR_Margin_Week_11cols` 的「融資 → 增減」；當週典型價 = (高 + 低 + 收) / 3，與資券資料用 ISO 年週對齊
* 期初融資部位與成本無法得知，所以這是散戶「期間內新增融資部位」的平均成本

In [4]:
# ===== 散戶（融資）買入平均成本 =====

MARGIN_FILE_TAG = "Margin_Week_11cols"
# 資券餘額 CSV 有兩個「增減」欄：第一個是融資、第二個是融券 (pandas 讀進來會改名為「增減.1」)
MARGIN_CHANGE_COLUMN = "增減"


def calc_retail_avg_cost(abbr, pivot_year_week, input_week_num, data_dir=None):
    """計算散戶 (融資) 在期間內的買進均價，回傳平均成本
    - 以 pivot_year_week (ISO 年, 第幾週) 為終點 (包含)，往前取 input_week_num 筆資券餘額週資料
    - 融資減少 (賣出還款) 不影響成本，只用融資增加的週：
      平均成本 = Σ(融資增加張數 × 當週典型價) / Σ(融資增加張數)；期間內融資都沒有增加時是 NaN"""
    margin = read_week_csv(abbr, MARGIN_FILE_TAG, data_dir)
    price = read_week_csv(abbr, ds.PRICE_FILE_TAG, data_dir)
    window = _week_window(margin, pivot_year_week, input_week_num, "資券餘額")
    lots = _to_number(window[MARGIN_CHANGE_COLUMN])  # 每週融資增減張數
    return _buy_avg_cost(window, lots, _week_typical_price(price, window), "融資")

### 函式定義(假設驗證：買進訊號回測)
假設：以錨點往前 `INPUT_WEEK_NUM` 筆資料做線性回歸，當
* `Major_Holders_Week` 的大戶籌碼比例斜率 > 0 (增加)
* `Securities_Lending_and_Borrowing_Week` 的借券賣出餘額斜率 < 0 (減少)
* `Margin_Week` 的融資餘額斜率 < 0 (減少)

時適合買進。驗證錨點之後 `OUTPUT_WEEK_NUM` 週的股價 (`Stock_Price_Week` 的最低價) 與同期間大戶買入平均成本的關係 (命中的定義見 `HIT_RULES`)，並與沒有訊號的樣本比較，看訊號是否真的有幫助。

In [12]:
# ===== 假設驗證（回測）：買進訊號出現後，未來 output_week_num 週的股價與大戶平均成本的關係 =====
# 買進訊號：錨點往前 input_week_num 筆資料做線性回歸，
#   大戶籌碼比例斜率 > 0、借券賣出餘額斜率 < 0、融資餘額斜率 < 0
# 驗證 (命中的定義見 HIT_RULES)：錨點之後 (不包含) output_week_num 週的最低價與大戶平均成本比較

MAJOR_HOLDERS_FILE_TAG = "Major_Holders_Week_5cols"
LENDING_FILE_TAG = "Securities_Lending_and_Borrowing_Week_8cols"
# 訊號名稱: (檔名標籤, 欄位, 斜率的方向 +1 / -1)
# 借券賣出餘額是借券 CSV 的第二個「餘額」欄 (pandas 改名為「餘額.1」)；融資餘額是資券餘額 CSV 的第一個「餘額」欄
SIGNAL_TRENDS = {
    "大戶籌碼": (MAJOR_HOLDERS_FILE_TAG, "大戶籌碼", 1),
    "借券賣出餘額": (LENDING_FILE_TAG, "餘額.1", -1),
    "融資餘額": (MARGIN_FILE_TAG, "餘額", -1),
}
MIN_TREND_POINTS = 3  # 線性回歸至少要有幾週的數值

# 命中的定義: (說明, 用 未來最低價 與 成本 判斷是否命中)
HIT_RULES = {
    "touch": ("未來最低價 <= 大戶成本：有機會用 <= 成本的價格買進", lambda low, cost: low <= cost),
    "hold": ("未來最低價 >= 大戶成本：股價守住大戶成本 (成本是支撐)", lambda low, cost: low >= cost),
}
DISTANCE_GROUPS = 5  # 「同錨點同距離差」把 錨點收盤價 / 成本 - 1 分成幾組 (依分位數)


def load_stock_week_data(abbr, data_dir=None):
    """讀取一檔股票的 5 份週資料，回傳 {檔名標籤: DataFrame}；只讀取現有的 csv，不會觸發 DataScrape
    資料有缺就回傳 None (這檔股票直接跳過)：缺少任何一份 csv、檔案讀不到 (例如正在寫入、剛被刪除)、
    內容無法解析，或欄位數與 DataScrape 的 WEEK_FILE_COLUMNS 不符
    每份資料都只有抓資料時已經結束的週 (read_week_csv 已排除還沒結束的週，不會修改 csv 檔)
    除了日期、年週以外的欄位都先轉成數值，之後計算就不用每次重新轉換"""
    folder = Path(data_dir or DATA_DIR) / abbr
    if any(_latest_week_file(folder, abbr, tag) is None for tag in ds.WEEK_FILE_COLUMNS):
        return None
    data = {}
    for tag, columns in ds.WEEK_FILE_COLUMNS.items():
        try:
            df = read_week_csv(abbr, tag, data_dir)
        except (ValueError, OSError):  # 內容無法解析，或檔案正在寫入、剛被刪除
            return None
        if df.shape[1] - 1 != len(columns):  # 欄位數不對 (最後一欄是 read_week_csv 加上的年週)
            return None
        for col in df.columns[1:-1]:  # 第一欄是日期、最後一欄是年週
            df[col] = _to_number(df[col])
        data[tag] = df
    return data


def _trend_slope(values):
    """線性回歸斜率 (values 由新到舊，先反轉成由舊到新)
    某幾週沒有數值時 (例如大戶籌碼在 2026/7/9 那週是「-」)，用其餘的週計算，x 仍是原本的週次位置；
    有數值的週少於 MIN_TREND_POINTS 時回傳 NaN"""
    y = _to_number(values).to_numpy(dtype=float)[::-1]
    x = np.arange(len(y))
    valid = ~np.isnan(y)
    if valid.sum() < MIN_TREND_POINTS:
        return float("nan")
    return np.polyfit(x[valid], y[valid], 1)[0]


def buy_signal_slopes(data, pivot_year_week, input_week_num):
    """回傳錨點往前 input_week_num 筆資料中，各訊號欄位的線性回歸斜率 {訊號名稱: 斜率}"""
    return {
        name: _trend_slope(_week_window(data[tag], pivot_year_week, input_week_num, name)[column])
        for name, (tag, column, _) in SIGNAL_TRENDS.items()
    }


def is_buy_signal(slopes):
    """大戶籌碼斜率 > 0、借券賣出餘額斜率 < 0、融資餘額斜率 < 0 時為買進訊號"""
    return all(np.sign(slopes[name]) == direction for name, (_, _, direction) in SIGNAL_TRENDS.items())


def backtest_buy_signal(input_week_nums, output_week_num, data_dir=None):
    """對 data_dir 中每一檔股票、每個 input_week_num、每個可用的錨點做回測，回傳 DataFrame (每列一個樣本)
    - 錨點要讓 5 份資料都有往前 input_week_num 筆，且之後有 output_week_num 週已結束的股價
      (還沒結束的週在 read_week_csv 讀取時已排除)
    - 缺少 5 份 csv 的股票會跳過；只讀取現有的 csv，不會觸發 DataScrape"""
    data_dir = Path(data_dir or DATA_DIR)
    rows, incomplete, errors = [], [], []
    for folder in sorted(p for p in data_dir.iterdir() if p.is_dir()):
        abbr = folder.name
        data = load_stock_week_data(abbr, data_dir)
        if data is None:
            incomplete.append(abbr)
            continue
        price = data[ds.PRICE_FILE_TAG]
        weeks = {tag: list(df["年週"]) for tag, df in data.items()}  # 各份資料的年週 (由新到舊)
        for n in input_week_nums:
            # 錨點必須出現在每一份資料中，所以只需要從其中一份 (資券餘額，筆數最少) 的年週裡找
            for pivot in weeks[MARGIN_FILE_TAG]:
                if not all(pivot in w and w.index(pivot) + n <= len(w) for w in weeks.values()):
                    continue
                p = weeks[ds.PRICE_FILE_TAG].index(pivot)
                future = price.iloc[max(p - output_week_num, 0) : p]  # 錨點之後已結束的週 (由新到舊)
                if len(future) < output_week_num or future[["高", "低", "收"]].isna().any().any():  # 週數不夠或股價有缺值
                    continue
                try:
                    slopes = buy_signal_slopes(data, pivot, n)
                    cost = _institutional_avg_cost(data[INSTITUTIONAL_FILE_TAG], price, pivot, n)
                except ValueError as e:  # 這個錨點的資料有問題 (例如某週沒有股價)，只跳過這個樣本
                    errors.append((abbr, n, pivot, str(e)))
                    continue
                rows.append({
                    "股票": abbr, "INPUT_WEEK_NUM": n, "錨點": pivot,
                    **{f"斜率_{k}": v for k, v in slopes.items()},
                    "訊號": is_buy_signal(slopes),
                    **{f"成本_{k}": v for k, v in cost.items()},
                    "錨點收盤": price["收"].iloc[p],
                    "未來最低": future["低"].min(),
                    "未來最高": future["高"].max(),
                    "未來收盤": future["收"].iloc[0],  # 未來最後一週的收盤價
                })
    print(f"回測股票 {len({r['股票'] for r in rows})} 檔，樣本 {len(rows)} 筆；跳過資料不完整的股票 {len(incomplete)} 檔、"
          f"資料有誤的樣本 {len(errors)} 筆 (分布在 {len({e[0] for e in errors})} 檔)")
    return pd.DataFrame(rows)


def _signal_diff(d, column, strata):
    """在 strata 欄位相同的每一組樣本內，計算 有訊號 - 沒訊號 的 column 平均，再依訊號數加權平均
    (例如 strata=["錨點"] 就是在同一段時間內比較，排除大盤漲跌的影響)"""
    diffs = [
        (p.loc[p["訊號"], column].mean() - p.loc[~p["訊號"], column].mean(), p["訊號"].sum())
        for _, p in d.groupby(strata) if p["訊號"].any() and not p["訊號"].all()
    ]
    return np.average([x for x, _ in diffs], weights=[w for _, w in diffs]) if diffs else np.nan


def summarize_backtest(results, hit_rule="touch"):
    """依 INPUT_WEEK_NUM 彙整回測結果，比較「有訊號」與「沒有訊號」的樣本 (只計算平均成本不是 NaN 的樣本)
    hit_rule 是 HIT_RULES 的 key：
    - 命中率：依 hit_rule 判斷未來 output_week_num 週的最低價與大戶平均成本
    - 同錨點差：每個錨點 (同一段時間) 先算 有訊號 - 沒訊號 的命中率，再依訊號數加權平均，排除大盤漲跌的影響
    - 同錨點同距離差：再把 錨點收盤價 / 成本 - 1 (股價離成本多遠) 分成 DISTANCE_GROUPS 組，同一組內比較；
      股價離成本越遠，本來就越不容易跌回成本，這樣才能看出命中率的差異是不是訊號造成的
    - 4週報酬差：錨點收盤價買進、持有到未來最後一週收盤的報酬，同錨點比較 有訊號 - 沒訊號
    - (touch) 需下跌命中率：錨點收盤價 > 成本 (要等股價跌下來才買得到) 的樣本中的命中率
    - (touch) 買進後報酬：命中時以成本價買進，到未來最後一週收盤的平均報酬"""
    is_hit = HIT_RULES[hit_rule][1]
    rows = []
    for n, g in results.groupby("INPUT_WEEK_NUM"):
        row = {"INPUT_WEEK_NUM": n, "樣本數": len(g), "錨點數": g["錨點"].nunique(), "訊號數": int(g["訊號"].sum())}
        for name in INSTITUTION_GROUPS:
            cost = f"成本_{name}"
            d = g[g[cost].notna()].copy()
            d["命中"] = is_hit(d["未來最低"], d[cost])
            d["距離"] = d["錨點收盤"] / d[cost] - 1
            d["距離分組"] = pd.qcut(d["距離"], DISTANCE_GROUPS, labels=False, duplicates="drop")
            d["4週報酬"] = d["未來收盤"] / d["錨點收盤"] - 1
            sig, none = d[d["訊號"]], d[~d["訊號"]]
            row[f"{name}_訊號命中率"] = sig["命中"].mean()
            row[f"{name}_無訊號命中率"] = none["命中"].mean()
            row[f"{name}_同錨點差"] = _signal_diff(d, "命中", ["錨點"])
            row[f"{name}_同錨點同距離差"] = _signal_diff(d, "命中", ["錨點", "距離分組"])
            row[f"{name}_訊號距離中位數"] = sig["距離"].median()
            row[f"{name}_無訊號距離中位數"] = none["距離"].median()
            row[f"{name}_4週報酬差"] = _signal_diff(d, "4週報酬", ["錨點"])
            if hit_rule == "touch":
                d["報酬"] = (d["未來收盤"] / d[cost] - 1).where(d["命中"])
                sig, none = d[d["訊號"]], d[~d["訊號"]]
                row[f"{name}_訊號需下跌命中率"] = sig.loc[sig["錨點收盤"] > sig[cost], "命中"].mean()
                row[f"{name}_無訊號需下跌命中率"] = none.loc[none["錨點收盤"] > none[cost], "命中"].mean()
                row[f"{name}_訊號買進後報酬"] = sig["報酬"].mean()
                row[f"{name}_無訊號買進後報酬"] = none["報酬"].mean()
        rows.append(row)
    return pd.DataFrame(rows).set_index("INPUT_WEEK_NUM")

### 函式定義(來回交易回測：買得比大戶成本低、賣得比散戶成本高)
* 計算：以錨點那一週 (包含) 往前 `INPUT_WEEK_NUM` 週的資料，算出大戶成本、散戶成本與各項特徵；兩個成本在之後的買賣中都固定不變
* 前提：錨點時 大戶成本 < 散戶成本，且價差 (散戶成本 / 大戶成本 - 1) >= `ROUND_TRIP_MIN_SPREAD` (3%)，成功時至少賺 3%
* 買進：錨點之後 `OUTPUT_WEEK_NUM` 週內，股價碰到大戶成本就買 (價格 <= 大戶成本)
* 賣出：從買進的**下一週**起 `OUTPUT_WEEK_NUM` 週內，股價碰到散戶成本就賣 (價格 >= 散戶成本)；沒碰到就在第 `OUTPUT_WEEK_NUM` 週收盤賣出，所以從錨點算起最長 8 週
* 停損：週收盤價跌破買價的 `ROUND_TRIP_STOP_LOSS` (10%) 就在當週收盤賣出；設成 `None` 就是不停損
* `ROUND_TRIP_PATTERNS`：提高來回成功率的模式。關鍵是「股價現在就在大戶成本附近」以及「價差相對於這檔股票平常一週的波動夠小」
* 分兩步計算：`collect_round_trip_samples` 讀資料、算成本與特徵 (最花時間，只做一次)；`apply_round_trip` 依買進期 / 賣出期 / 停損模擬交易，同一批樣本可以套用不同的交易規則 (例如下面的交易模式)
* 只用完整的週：每份 csv 依檔名的抓取日期，排除抓資料時還沒結束的那一週 (週一 ~ 週五抓的當週)；只是讀取時排除，不會修改 csv 檔

In [6]:
# ===== 來回交易回測：買進價格 <= 大戶成本、之後賣出價格 >= 散戶成本 =====
# 前提：錨點時 大戶成本 < 散戶成本 (才有價差可賺)
# 買進：錨點之後 buy_weeks 週內，最低價碰到大戶成本就買 (當週開盤已低於成本時用開盤價)
# 賣出：買進之後 hold_weeks 週內，最高價碰到散戶成本就賣 (當週開盤已高於成本時用開盤價)；
#       沒碰到就在第 hold_weeks 週收盤賣出
# 停損 (stop_loss，可不設)：從買進當週起，週收盤價跌破 買價 × (1 - stop_loss) 就在當週收盤賣出
# 分兩步計算：collect_round_trip_samples 讀資料、算成本與特徵 (最花時間，只做一次)；
#             apply_round_trip 依買進期 / 賣出期 / 停損模擬交易 (很快，同一批樣本可以套用很多組交易規則)
# 還沒結束的週在 read_week_csv 讀取時已排除，所以錨點、錨點往前的資料與錨點之後的股價都是完整的週

ROUND_TRIP_COSTS = {"外資": "外資", "投信自營": "投信+自營商"}  # 回測用的大戶成本: INSTITUTION_GROUPS 的 key
NO_TRADE = (False, False, False, np.nan, np.nan, np.nan, np.nan)  # 沒有買到 (或不符合前提) 時的交易結果


def _window_change(values, percent_points):
    """window 期間 (由新到舊) 用線性回歸算出的變化量：
    percent_points=True 用於百分比欄位，回傳變化幾個百分點；否則回傳相對平均值的變化比例"""
    y = _to_number(values).to_numpy(dtype=float)[::-1]
    x = np.arange(len(y))
    valid = ~np.isnan(y)
    if valid.sum() < MIN_TREND_POINTS:
        return np.nan
    change = np.polyfit(x[valid], y[valid], 1)[0] * (len(y) - 1)
    if percent_points:
        return change
    mean = np.abs(y[valid]).mean()
    return change / mean if mean else np.nan


def simulate_round_trip(future, buy_price, sell_price, buy_weeks, hold_weeks, stop_loss=None):
    """future 是錨點之後的股價週資料，每列是 [開, 高, 低, 收] (numpy array，由舊到新，至少 buy_weeks + hold_weeks 週)
    stop_loss：週收盤價跌破 買價 × (1 - stop_loss) 就在當週收盤賣出，例如 0.10；None = 不停損
    回傳 (是否買到, 是否賣到, 是否停損, 買價, 賣價, 錨點後第幾週買到, 錨點後第幾週賣出)"""
    for i in range(buy_weeks):
        week_open, _, week_low, week_close = future[i]
        if week_low <= buy_price:
            fill = min(buy_price, week_open)
            stop_price = fill * (1 - stop_loss) if stop_loss else -np.inf
            if week_close <= stop_price:  # 買進當週就收在停損價以下
                return True, False, True, fill, week_close, i + 1, i + 1
            for j in range(i + 1, i + 1 + hold_weeks):  # 買進的下一週才開始賣
                week_open, week_high, _, week_close = future[j]
                if week_high >= sell_price:  # 盤中碰到散戶成本就成交，比週收盤的停損判斷早
                    return True, True, False, fill, max(sell_price, week_open), i + 1, j + 1
                if week_close <= stop_price:
                    return True, False, True, fill, week_close, i + 1, j + 1
            return True, False, False, fill, future[i + hold_weeks][3], i + 1, i + 1 + hold_weeks
    return NO_TRADE


def round_trip_features(data, pivot_year_week, input_week_num):
    """錨點往前 input_week_num 筆資料的籌碼、股價特徵 (f_ 開頭)：
    百分比欄位是變化幾個百分點，其他是相對變化比例，買超強度 = 期間買超張數 / 成交量，
    週波動率 = 每週 (最高 - 最低) / 收盤 的平均"""
    w = {tag: _week_window(df, pivot_year_week, input_week_num, tag) for tag, df in data.items()}
    inst, major, margin, prices = w[INSTITUTIONAL_FILE_TAG], w[MAJOR_HOLDERS_FILE_TAG], w[MARGIN_FILE_TAG], w[ds.PRICE_FILE_TAG]
    close = prices["收"]
    volume = inst["成交量"].sum()
    return {
        "f_大戶籌碼": _window_change(major["大戶籌碼"], True),
        "f_外資持股": _window_change(major["外資籌碼"], True),
        "f_董監持股": _window_change(major["董監持股"], True),
        "f_借券賣出餘額": _window_change(w[LENDING_FILE_TAG]["餘額.1"], False),
        "f_融資餘額": _window_change(margin["餘額"], False),
        "f_融資使用率": _window_change(margin["使用率％"], True),
        "f_融券餘額": _window_change(margin["餘額.1"], False),
        "f_券資比": _window_change(margin["券資比"], True),
        "f_外資買超強度": inst["外資(張)"].sum() / volume if volume else np.nan,
        "f_投信自營買超強度": (inst["投信(張)"] + inst["自營商(張)"]).sum() / volume if volume else np.nan,
        "f_股價動能": close.iloc[0] / close.iloc[-1] - 1,
        "f_近4週動能": close.iloc[0] / close.iloc[min(3, len(close) - 1)] - 1,
        "f_成交量變化": _window_change(inst["成交量"], False),
        "f_週波動率": ((prices["高"] - prices["低"]) / close).mean(),
    }


def _valid_pivots(weeks, input_week_num):
    """5 份資料都有、而且往前都有 input_week_num 筆資料的錨點週 (由新到舊)；weeks 是 {檔名標籤: 年週 list (由新到舊)}"""
    return [pivot for pivot in weeks[MARGIN_FILE_TAG]  # 資券餘額的筆數最少，從它的年週裡找就夠了
            if all(pivot in w and w.index(pivot) + input_week_num <= len(w) for w in weeks.values())]


def _round_trip_sample(abbr, data, pivot, input_week_num):
    """一檔股票在錨點 pivot 的大戶/散戶成本與特徵 (一列樣本，不含交易結果)；這個錨點的資料有問題時 raise ValueError
    每個大戶成本 (ROUND_TRIP_COSTS) 另外算：
    - f_價差 = 散戶成本 / 大戶成本 - 1 (兩邊都照成本價成交時的獲利)
    - f_需要漲幅 = 散戶成本 / min(大戶成本, 錨點收盤) - 1 (從預計的買價要漲多少才賣得到)
    - f_價差波動比 = f_價差 / f_週波動率 (價差相當於幾週的平均波動)
    - f_收盤位置 = (錨點收盤 - 大戶成本) / (散戶成本 - 大戶成本)：< 0 低於大戶成本，0 ~ 1 在兩個成本之間，> 1 高於散戶成本
    - 前提 = 兩個成本都算得出來，且大戶成本 < 散戶成本"""
    price = data[ds.PRICE_FILE_TAG]
    big = _institutional_avg_cost(data[INSTITUTIONAL_FILE_TAG], price, pivot, input_week_num)
    margin = _week_window(data[MARGIN_FILE_TAG], pivot, input_week_num, "資券餘額")
    retail = _buy_avg_cost(margin, margin[MARGIN_CHANGE_COLUMN], _week_typical_price(price, margin), "融資")
    features = round_trip_features(data, pivot, input_week_num)
    close = price["收"].iloc[list(price["年週"]).index(pivot)]
    row = {"股票": abbr, "錨點": pivot, "錨點收盤": close, "成本_散戶": retail, **features}
    for name, group in ROUND_TRIP_COSTS.items():
        cost = big[group]
        ok = not np.isnan(cost) and not np.isnan(retail) and cost < retail
        spread = retail / cost - 1
        row.update({
            f"成本_{name}": cost,
            f"f_距{name}成本": close / cost - 1,
            f"f_價差_{name}": spread,
            f"f_需要漲幅_{name}": retail / np.minimum(cost, close) - 1,
            f"f_價差波動比_{name}": spread / row["f_週波動率"] if row["f_週波動率"] else np.nan,
            f"f_收盤位置_{name}": (close - cost) / (retail - cost) if ok else np.nan,
            f"前提_{name}": ok,
        })
    return row


def collect_round_trip_samples(input_week_num, max_future_weeks, min_future_weeks=1, data_dir=None):
    """對 data_dir 中每一檔股票、每個可用的錨點，計算大戶/散戶成本與特徵 (見 _round_trip_sample)，回傳 DataFrame (每列一個樣本)
    「未來股價」欄是錨點之後已經結束的週 [開, 高, 低, 收] (由舊到新，最多 max_future_weeks 週)，
    少於 min_future_weeks 週的錨點不列入；交易結果由 apply_round_trip 計算
    缺少 5 份 csv 的股票會跳過，不會觸發 DataScrape"""
    data_dir = Path(data_dir or DATA_DIR)
    rows, skipped = [], 0
    for folder in sorted(p for p in data_dir.iterdir() if p.is_dir()):
        abbr = folder.name
        data = load_stock_week_data(abbr, data_dir)
        if data is None:
            skipped += 1
            continue
        price = data[ds.PRICE_FILE_TAG]
        weeks = {tag: list(df["年週"]) for tag, df in data.items()}
        for pivot in _valid_pivots(weeks, input_week_num):
            p = weeks[ds.PRICE_FILE_TAG].index(pivot)
            future = price.iloc[max(p - max_future_weeks, 0) : p].iloc[::-1]  # 錨點之後已結束的週 (由舊到新)
            if len(future) < min_future_weeks:
                continue
            try:
                row = _round_trip_sample(abbr, data, pivot, input_week_num)
            except ValueError:  # 這個錨點的資料有問題 (例如某週沒有股價)，只跳過這個樣本
                continue
            row["未來股價"] = future[["開", "高", "低", "收"]].to_numpy(dtype=float)
            rows.append(row)
    print(f"來回交易樣本：股票 {len({r['股票'] for r in rows})} 檔，樣本 {len(rows)} 筆；跳過資料不完整的股票 {skipped} 檔")
    return pd.DataFrame(rows)


def apply_round_trip(samples, buy_weeks, hold_weeks, stop_loss=None):
    """用 samples (collect_round_trip_samples 的結果) 中錨點之後至少有 buy_weeks + hold_weeks 週、而且這幾週股價沒有缺值的樣本
    模擬來回交易 (規則見 simulate_round_trip)，回傳這些樣本，並加上每個大戶成本的結果欄位：
    買到_、賣到_、停損_、報酬_ (賣價 / 買價 - 1，未扣交易成本)、持有週數_ (買進後第幾週賣出)、出場週_ (錨點後第幾週賣出)"""
    weeks = buy_weeks + hold_weeks
    d = samples.loc[[len(f) >= weeks and not np.isnan(f[:weeks]).any() for f in samples["未來股價"]]].copy()
    for name in ROUND_TRIP_COSTS:
        trades = pd.DataFrame(
            [simulate_round_trip(future, cost, retail, buy_weeks, hold_weeks, stop_loss) if ok else NO_TRADE
             for future, cost, retail, ok in zip(d["未來股價"], d[f"成本_{name}"], d["成本_散戶"], d[f"前提_{name}"])],
            index=d.index, columns=["買到", "賣到", "停損", "買價", "賣價", "買進週", "出場週"],
        )
        for column in ["買到", "賣到", "停損"]:
            d[f"{column}_{name}"] = trades[column].astype(bool)
        d[f"報酬_{name}"] = trades["賣價"] / trades["買價"] - 1
        d[f"持有週數_{name}"] = trades["出場週"] - trades["買進週"]
        d[f"出場週_{name}"] = trades["出場週"]
    return d


def backtest_round_trip(input_week_num, buy_weeks, hold_weeks, data_dir=None, stop_loss=None):
    """只用一組交易規則時的簡便寫法 = collect_round_trip_samples + apply_round_trip"""
    weeks = buy_weeks + hold_weeks
    return apply_round_trip(collect_round_trip_samples(input_week_num, weeks, weeks, data_dir), buy_weeks, hold_weeks, stop_loss)


# ===== 提高來回成功率的模式 (門檻來自 DataCollection 的分析，INPUT_WEEK_NUM = 12、OUTPUT_WEEK_NUM = 4) =====
ROUND_TRIP_MIN_SPREAD = 0.03  # 價差 (散戶成本 / 大戶成本 - 1) 至少 3%：成功時至少賺 3%
# 建議的停損 (A：4 週模式)：週收盤價跌破買價 10% 就賣出。回測中，對「+ 外資持股沒有增加」的兩個模式幾乎不影響成功率與平均報酬，
# 只切掉最差的幾筆；對其他模式會稍微降低平均報酬。盤中觸價停損 (3% ~ 15%) 會被正常波動洗出場，成功率與平均報酬都下降。
# 週收盤停損擋不住一週內的急跌 (例如 2026 年第 45 週)，這種風險要靠每檔金額小、持有時間短來控制
ROUND_TRIP_STOP_LOSS = 0.10
MIN_WEEK_SAMPLES = 5  # 「最差一週來回成功率」只看符合樣本數至少這麼多的錨點週


def _near_cost(x, name, max_ratio, low, high):
    """價差不超過 max_ratio 週的平均波動，且錨點收盤價在大戶成本的 low ~ high 之間 (例如 -3% ~ +5%)"""
    distance = x[f"f_距{name}成本"]
    return (x[f"f_價差波動比_{name}"] <= max_ratio) & (distance >= low) & (distance <= high)


# C 模式 (精選) 的候選條件：平衡 + 外資持股沒有增加，再加上「價差 / 週波動」的上限。
# 原本是「每個錨點週挑價差 / 週波動最小的 2 檔」，單一股票無法這樣判斷，所以改用上限：
# 0.26 約等於 2026 年第 29 ~ 42 週，各錨點週第 2 小的「價差 / 週波動」的中位數 (回測結果見「交易模式比較」)
MODE_C_MAX_RATIO = 0.26
MODE_C_DISTANCE = (-0.03, 0.05)  # 錨點收盤價距大戶成本的範圍 (-3% ~ +5%)
MODE_C_PATTERN = f"精選：價差 <= {MODE_C_MAX_RATIO} 週波動、距大戶成本 -3% ~ +5% + 外資持股沒有增加"

# 模式名稱 -> (df, 大戶成本名稱) 回傳是否符合
ROUND_TRIP_PATTERNS = {
    "原本訊號 (大戶籌碼↑ 借券↓ 融資↓)": lambda x, name: (x["f_大戶籌碼"] > 0) & (x["f_借券賣出餘額"] < 0) & (x["f_融資餘額"] < 0),
    "寬鬆：價差 <= 0.5 週波動、距大戶成本 -5% ~ +5%": lambda x, name: _near_cost(x, name, 0.5, -0.05, 0.05),
    "平衡：價差 <= 0.4 週波動、距大戶成本 -3% ~ +5%": lambda x, name: _near_cost(x, name, 0.4, -0.03, 0.05),
    "嚴格：價差 <= 0.3 週波動、距大戶成本 -3% ~ +5%": lambda x, name: _near_cost(x, name, 0.3, -0.03, 0.05),
    "平衡 + 外資持股沒有增加": lambda x, name: _near_cost(x, name, 0.4, -0.03, 0.05) & (x["f_外資持股"] <= 0),
    "嚴格 + 外資持股沒有增加": lambda x, name: _near_cost(x, name, 0.3, -0.03, 0.05) & (x["f_外資持股"] <= 0),
    MODE_C_PATTERN: lambda x, name: _near_cost(x, name, MODE_C_MAX_RATIO, *MODE_C_DISTANCE) & (x["f_外資持股"] <= 0),
}


def _round_trip_stats(x, name, n_weeks, early_weeks):
    """一組樣本的來回交易統計 (定義見「來回交易回測」說明)；n_weeks 是全部的錨點週數，early_weeks 是前期的錨點週"""
    sold, bought = x[f"賣到_{name}"], x[f"買到_{name}"]
    returns = x[f"報酬_{name}"]
    early = x["錨點"].isin(early_weeks)
    by_week = x.groupby("錨點")[f"賣到_{name}"].agg(["mean", "size"])
    return {
        "機會數": len(x),
        "平均每週機會數": len(x) / n_weeks,
        "價差中位數": x[f"f_價差_{name}"].median(),
        "買到率": bought.mean(),
        "買到後成功率": sold[bought].mean(),
        "來回成功率": sold.mean(),
        "前期來回成功率": sold[early].mean(),
        "後期來回成功率": sold[~early].mean(),
        "最差一週來回成功率": by_week.loc[by_week["size"] >= MIN_WEEK_SAMPLES, "mean"].min(),
        "停損出場率": x[f"停損_{name}"][bought].mean(),
        "成功時平均獲利": returns[sold].mean(),
        "未成功時平均報酬": returns[bought & ~sold].mean(),
        "最差5%報酬": returns[bought].quantile(0.05),
        "買到後平均報酬": returns[bought].mean(),
        "成功平均持有週數": x.loc[sold, f"持有週數_{name}"].mean(),
    }


def summarize_round_trip(results, name):
    """比較各模式與「基準」(大戶成本 < 散戶成本、價差 >= ROUND_TRIP_MIN_SPREAD 的全部機會) 的來回交易結果"""
    d = results[results[f"前提_{name}"] & (results[f"f_價差_{name}"] >= ROUND_TRIP_MIN_SPREAD)]
    weeks = sorted(d["錨點"].unique())
    early_weeks = weeks[: len(weeks) // 2]  # 前一半的錨點週為前期，其餘為後期
    rows = [{"模式": f"基準：價差 >= {ROUND_TRIP_MIN_SPREAD:.0%} 的全部機會", **_round_trip_stats(d, name, len(weeks), early_weeks)}]
    for pattern, rule in ROUND_TRIP_PATTERNS.items():
        rows.append({"模式": pattern, **_round_trip_stats(d[rule(d, name).fillna(False)], name, len(weeks), early_weeks)})
    return pd.DataFrame(rows).set_index("模式")

### 函式定義(交易模式：候選條件 + 交易規則)
大戶成本都用外資成本 (`MODE_COST`)，三個模式的差別 (`TRADING_MODES`)：

| 模式 | 候選條件 | 買進期 | 賣出期 | 停損 |
|---|---|---|---|---|
| A：4 週模式 (原本) | 平衡 + 外資持股沒有增加 | 錨點後 4 週 | 買進後 4 週 | 週收盤跌破買價 10% |
| B：2 週模式 | 平衡 + 外資持股沒有增加 | 錨點後 2 週 | 買進後 2 週 | 不停損 |
| C：精選 2 週模式 (推薦) | 平衡 + 外資持股沒有增加，而且「價差 / 週波動」<= `MODE_C_MAX_RATIO` (0.26) | 錨點後 2 週 | 買進後 2 週 | 不停損 |

C 原本是「每個錨點週挑價差 / 週波動最小的 2 檔」，單一股票無法這樣判斷，所以改用上限 0.26：約等於 2026 年第 29 ~ 42 週，各錨點週第 2 小的「價差 / 週波動」的中位數。

C 的交易流程 (錨點週 W)：
1. W 結束後：用最近 `INPUT_WEEK_NUM` 週 (含 W) 的資料，算出外資成本、散戶成本、週波動率與外資持股的回歸斜率，同時符合以下條件才買
   * 外資成本 < 散戶成本，且價差 (散戶成本 / 外資成本 - 1) >= 3%
   * 價差 / 週波動率 <= 0.26
   * W 的收盤價在外資成本的 -3% ~ +5% 之間
   * 外資持股斜率 <= 0
2. 同一週符合的股票太多、資金不夠時，先買「價差 / 週波動」比較小的
3. W+1、W+2 (買進期)：以外資成本掛限價買單，開盤就低於外資成本時以開盤價成交；2 週都沒成交就放棄，不追價
4. 買進的下一週起 2 週 (賣出期)：以散戶成本掛限價賣單，開盤就高於散戶成本時以開盤價成交；買進當週不賣
5. 2 週內沒賣到，就在第 2 週週五收盤賣出
6. 每檔投入相同金額；不用盤中觸價停損 (這些股票波動大，常先跌 10% 以上再漲回散戶成本)

In [7]:
# ===== 交易模式 = 候選條件 + 每週挑幾檔 + 交易規則 (門檻來自 DataCollection 的分析，INPUT_WEEK_NUM = 12) =====
ROUND_TRIP_TRADE_COST = 0.001425 * 2 + 0.003  # 一次來回的交易成本：手續費 0.1425% × 2 (未折扣) + 證交稅 0.3%
MODE_COST = "外資"  # 交易模式用的大戶成本 (ROUND_TRIP_COSTS 的 key)；改用投信自營成本時結果明顯較差

# pattern：候選條件 (ROUND_TRIP_PATTERNS 的 key)
# top_k：每個錨點週只買「價差 / 週波動」最小的前幾檔；None = 全部候選都買
# buy_weeks：錨點之後幾週內掛大戶成本買進；hold_weeks：買進的下一週起幾週內掛散戶成本賣出，沒賣到就在最後一週收盤賣出
# stop_loss：週收盤價跌破 買價 × (1 - stop_loss) 就在當週收盤賣出；None = 不停損
TRADING_MODES = {
    "A：4 週模式 (原本)": {"pattern": "平衡 + 外資持股沒有增加", "top_k": None, "buy_weeks": 4, "hold_weeks": 4, "stop_loss": 0.10},
    "B：2 週模式": {"pattern": "平衡 + 外資持股沒有增加", "top_k": None, "buy_weeks": 2, "hold_weeks": 2, "stop_loss": None},
    "C：精選 2 週模式 (推薦)": {"pattern": MODE_C_PATTERN, "top_k": None, "buy_weeks": 2, "hold_weeks": 2, "stop_loss": None},
}
MODE_C_NAME = "C：精選 2 週模式 (推薦)"  # 本週建議買入、個股 C 模式檢查用的模式


def _round_trip_universe(results, name):
    """大戶成本 < 散戶成本，且價差 >= ROUND_TRIP_MIN_SPREAD 的全部機會"""
    return results[results[f"前提_{name}"] & (results[f"f_價差_{name}"] >= ROUND_TRIP_MIN_SPREAD)]


def select_mode_trades(results, name, pattern, top_k=None):
    """results (apply_round_trip 的結果) 中符合 pattern 的機會；
    top_k 不是 None 時，每個錨點週只留「價差 / 週波動」最小的 top_k 檔 (從該週全部的候選中挑)"""
    d = _round_trip_universe(results, name)
    x = d[ROUND_TRIP_PATTERNS[pattern](d, name).fillna(False)]
    if top_k:
        x = x.sort_values(f"f_價差波動比_{name}", kind="stable").groupby("錨點").head(top_k)
    return x


def _mode_stats(x, name, anchors):
    """一個交易模式的統計 (報酬已扣 ROUND_TRIP_TRADE_COST)；anchors 是可以回測的錨點週 (由舊到新)"""
    if not anchors:
        return {"錨點週數": 0, "機會數": len(x)}
    bought, sold = x[f"買到_{name}"], x[f"賣到_{name}"]
    net = x.loc[bought, f"報酬_{name}"] - ROUND_TRIP_TRADE_COST
    by_week = net.groupby(x.loc[bought, "錨點"]).mean()
    (first_year, first_week), (last_year, last_week) = anchors[0], anchors[-1]
    return {
        "錨點範圍": f"{first_year}W{first_week} ~ {last_year}W{last_week}",
        "錨點週數": len(anchors),
        "機會數": len(x),
        "平均每週機會數": len(x) / len(anchors),
        "買到率": bought.mean(),
        "來回成功率": sold.mean(),
        "買到後成功率": sold[bought].mean(),
        "停損出場率": x.loc[bought, f"停損_{name}"].mean(),
        "平均淨報酬": net.mean(),
        "虧損率": (net < 0).mean(),
        "虧損時平均": net[net < 0].mean(),
        "最差一筆": net.min(),
        "賺錢週": f"{(by_week > 0).sum()}/{len(by_week)}",
        "平均資金占用週數": (x.loc[bought, f"持有週數_{name}"] + 1).mean(),
    }


def summarize_trading_modes(samples, name=MODE_COST, modes=None):
    """比較交易模式 (預設 TRADING_MODES)，samples 是 collect_round_trip_samples 的結果，回傳 (全部, 同期) 兩張表：
    - 全部：每個模式用它能回測的所有樣本；2 週模式只需要錨點後 4 週的股價，所以比 4 週模式多算最近幾個錨點週
    - 同期：只看每個模式都能回測的樣本 (股票 × 錨點週)，用來公平比較；精選的 top_k 仍是從該週全部的候選中挑"""
    runs = {}
    for mode, m in (modes or TRADING_MODES).items():
        results = apply_round_trip(samples, m["buy_weeks"], m["hold_weeks"], m["stop_loss"])
        runs[mode] = (results, select_mode_trades(results, name, m["pattern"], m["top_k"]))
    common = set.intersection(*(set(zip(results["股票"], results["錨點"])) for results, _ in runs.values()))
    tables = {"全部": [], "同期": []}
    for mode, (results, trades) in runs.items():
        universe = _round_trip_universe(results, name)
        for label, keep in (("全部", None), ("同期", common)):
            u, t = universe, trades
            if keep is not None:
                u = u.loc[[key in keep for key in zip(u["股票"], u["錨點"])]]
                t = t.loc[[key in keep for key in zip(t["股票"], t["錨點"])]]
            tables[label].append({"模式": mode, **_mode_stats(t, name, sorted(u["錨點"].unique()))})
    return tuple(pd.DataFrame(rows).set_index("模式") for rows in tables.values())


# ===== 本週建議買入：每檔股票用最新且完整的週當錨點，依交易模式列出建議買入的股票 =====


def _tick_size(price, is_etf=False):
    """台股的升降單位 (股價最小的跳動)：股票 (上市、上櫃相同) 與 ETF (代碼 00 開頭) 不同"""
    if is_etf:
        return 0.01 if price < 50 else 0.05
    for limit, tick in ((10, 0.01), (50, 0.05), (100, 0.1), (500, 0.5), (1000, 1)):
        if price < limit:
            return tick
    return 5


def _round_to_tick(price, direction, is_etf=False):
    """把成本調整成可以掛單的價格：買進 direction="down" 往下取 (不高於大戶成本)，賣出 "up" 往上取 (不低於散戶成本)"""
    tick = _tick_size(price, is_etf)
    steps = np.floor(price / tick + 1e-9) if direction == "down" else np.ceil(price / tick - 1e-9)
    return round(float(steps * tick), 2)


RECOMMENDATION_COLUMNS = ["股票", "錨點週", "錨點收盤", "大戶成本(買入價)", "散戶成本(賣出價)", "價差(%)", "週波動(%)",
                          "價差/週波動", "外資持股斜率(百分點/週)", "買進掛單價", "賣出掛單價", "買進期間", "買進期狀態"]


def _recommendation_table(x, name, input_week_num, buy_weeks, today):
    """recommend_trades 的輸出表：最新的錨點週在前，同一週依「價差 / 週波動」由小到大 (越前面越優先)"""
    rows = []
    for _, r in x.iterrows():
        monday = _week_monday(r["錨點"])
        start = monday + timedelta(weeks=1)                   # 錨點下一週的週一
        end = monday + timedelta(weeks=buy_weeks, days=4)     # 錨點後第 buy_weeks 週的週五
        is_etf = r["股票"].startswith("00")
        rows.append({
            "股票": r["股票"],
            "錨點週": f"{r['錨點'][0]}W{r['錨點'][1]:02d}",
            "錨點收盤": r["錨點收盤"],
            "大戶成本(買入價)": r[f"成本_{name}"],
            "散戶成本(賣出價)": r["成本_散戶"],
            "價差(%)": r[f"f_價差_{name}"] * 100,
            "週波動(%)": r["f_週波動率"] * 100,
            "價差/週波動": r[f"f_價差波動比_{name}"],
            "外資持股斜率(百分點/週)": r["f_外資持股"] / (input_week_num - 1),  # f_外資持股 = 斜率 × (週數 - 1)
            "買進掛單價": _round_to_tick(r[f"成本_{name}"], "down", is_etf),
            "賣出掛單價": _round_to_tick(r["成本_散戶"], "up", is_etf),
            "買進期間": f"{start:%m/%d} ~ {end:%m/%d}",
            "買進期狀態": "尚未開始" if today < start else "已結束" if today > end else "進行中",
        })
    table = pd.DataFrame(rows, columns=RECOMMENDATION_COLUMNS)
    table = table.sort_values(["錨點週", "價差/週波動"], ascending=[False, True]).reset_index(drop=True)
    return table.round({"大戶成本(買入價)": 2, "散戶成本(賣出價)": 2, "價差(%)": 2, "週波動(%)": 2,
                        "價差/週波動": 3, "外資持股斜率(百分點/週)": 4})


def recommend_trades(mode="C：精選 2 週模式 (推薦)", input_week_num=12, data_dir=None, name=MODE_COST,
                     max_anchor_lag_weeks=1, today=None):
    """每檔股票用「最新且 5 份資料都完整的週」當錨點 (還沒結束的週在讀取時已排除，所以各股的錨點週可能不同)，
    依 TRADING_MODES[mode] 的候選條件 (pattern) 與每週挑幾檔 (top_k) 列出建議買入的股票；不使用停損
    - 錨點比全部股票中最新的錨點週早超過 max_anchor_lag_weeks 週的股票 (資料太舊，例如這次沒有更新到) 不列入
    - today：判斷買進期狀態用的日期，預設是今天
    回傳 (建議買入, 全部候選) 兩張表，欄位說明見「本週建議買入」；只讀取現有的 csv，不會觸發 DataScrape"""
    m = TRADING_MODES[mode]
    today = today or datetime.now().date()
    data_dir = Path(data_dir or DATA_DIR)
    rows, incomplete, no_pivot = [], 0, 0
    for folder in sorted(p for p in data_dir.iterdir() if p.is_dir()):
        data = load_stock_week_data(folder.name, data_dir)
        if data is None:
            incomplete += 1
            continue
        pivots = _valid_pivots({tag: list(df["年週"]) for tag, df in data.items()}, input_week_num)
        try:
            rows.append(_round_trip_sample(folder.name, data, pivots[0], input_week_num))  # pivots 由新到舊
        except (IndexError, ValueError):  # 沒有可用的錨點，或最新錨點的資料有問題
            no_pivot += 1
    if not rows:
        raise ValueError(f"{data_dir} 中沒有可以分析的股票")
    samples = pd.DataFrame(rows)
    newest = _week_monday(max(samples["錨點"]))
    recent = [(newest - _week_monday(anchor)).days // 7 <= max_anchor_lag_weeks for anchor in samples["錨點"]]
    stale = len(samples) - sum(recent)
    samples = samples.loc[recent]
    candidates = select_mode_trades(samples, name, m["pattern"])
    picks = select_mode_trades(samples, name, m["pattern"], m["top_k"])
    counts = samples["錨點"].value_counts().sort_index(ascending=False)
    print("各錨點週可分析的股票：" + "、".join(f"{year}W{week:02d} {n} 檔" for (year, week), n in counts.items()))
    limit = f" (每個錨點週最多 {m['top_k']} 檔)" if m["top_k"] else ""
    print(f"略過：資料不齊全 {incomplete} 檔、沒有可用的錨點 {no_pivot} 檔、資料太舊 {stale} 檔；"
          f"符合「{m['pattern']}」的候選 {len(candidates)} 檔，建議買入 {len(picks)} 檔{limit}")
    return (_recommendation_table(picks, name, input_week_num, m["buy_weeks"], today),
            _recommendation_table(candidates, name, input_week_num, m["buy_weeks"], today))


# ===== 個股 C 模式檢查：輸入一檔股票，用它最新的完整週判斷是否符合 C 模式 =====


def _mode_c_conditions(row, name, input_week_num):
    """C 模式的每個條件 (條件, 數值, 要求, 是否符合)，與 _round_trip_universe 和 MODE_C_PATTERN 的條件相同"""
    cost, retail = row[f"成本_{name}"], row["成本_散戶"]
    spread, ratio, distance = row[f"f_價差_{name}"], row[f"f_價差波動比_{name}"], row[f"f_距{name}成本"]
    low, high = MODE_C_DISTANCE
    return [
        (f"大戶成本 ({name}) / 散戶成本", f"{cost:.2f} / {retail:.2f}", "大戶成本 < 散戶成本", bool(row[f"前提_{name}"])),
        ("價差", f"{spread:.2%}", f">= {ROUND_TRIP_MIN_SPREAD:.0%}", bool(spread >= ROUND_TRIP_MIN_SPREAD)),
        ("價差 / 週波動", f"{ratio:.3f}", f"<= {MODE_C_MAX_RATIO}", bool(ratio <= MODE_C_MAX_RATIO)),
        ("錨點收盤距大戶成本", f"{distance:+.2%}", f"{low:+.0%} ~ {high:+.0%}", bool(low <= distance <= high)),
        ("外資持股斜率 (百分點/週)", f"{row['f_外資持股'] / (input_week_num - 1):+.4f}", "<= 0", bool(row["f_外資持股"] <= 0)),
    ]


# 個股 C 模式檢查只讀取 csv；資料沒有、不完整或太舊時，提醒到輸入代碼那一格 (ensure_week_data) 重新抓取
UPDATE_HINT = "請先在「輸入公司代碼，檢查週資料」那一格輸入同一個代碼，用 DataScrape 抓取最新資料"


def check_stock_mode_c(abbr, input_week_num=12, data_dir=None, name=MODE_COST, max_lag_weeks=1, today=None):
    """個股 C 模式檢查，只讀取現有的 csv，不會觸發 DataScrape：
    1. 前提：資料夾內 5 份 csv 完整，而且最新的完整週距離今天所在的週不超過 max_lag_weeks 週
       (預設 1 週，相差 >= 2 週就算太舊)；不通過就印出提醒並結束，回傳 None
    2. 以最新的完整週當錨點，逐項檢查 C 模式 (TRADING_MODES[MODE_C_NAME]) 的條件，
       回傳 (條件表, 是否符合, 交易計畫表)；不符合時交易計畫表是 None"""
    mode = TRADING_MODES[MODE_C_NAME]
    today = today or datetime.now().date()
    data_dir = Path(data_dir or DATA_DIR)
    folder = data_dir / abbr
    if not folder.is_dir():
        print(f"⚠ 找不到資料夾「{abbr}」，{UPDATE_HINT}")
        return None
    problems = check_week_files(folder, abbr)
    data = None if problems else load_stock_week_data(abbr, data_dir)
    if data is None:
        print(f"⚠ 資料夾「{abbr}」的 5 份 csv 不完整，{UPDATE_HINT}：")
        for problem in problems or ["有 csv 讀不到 (例如正在寫入) 或欄位數不對"]:
            print(f"   - {problem}")
        return None
    pivots = _valid_pivots({tag: list(df["年週"]) for tag, df in data.items()}, input_week_num)
    if not pivots:
        print(f"⚠ 「{abbr}」5 份 csv 共同的完整週不到 {input_week_num} 週，無法計算")
        return None
    anchor, this_week = pivots[0], today.isocalendar()[:2]  # pivots 由新到舊，第一個就是最新的完整週
    lag = (_week_monday(this_week) - _week_monday(anchor)).days // 7
    if lag > max_lag_weeks:
        print(f"⚠ 「{abbr}」最新的完整週是 {anchor[0]}W{anchor[1]:02d}，今天在 {this_week[0]}W{this_week[1]:02d}，"
              f"相差 {lag} 週 (>= {max_lag_weeks + 1} 週)，資料太舊，{UPDATE_HINT}")
        return None
    try:
        row = _round_trip_sample(abbr, data, anchor, input_week_num)
    except ValueError as e:
        print(f"⚠ 「{abbr}」錨點週 {anchor[0]}W{anchor[1]:02d} 的資料有問題：{e}")
        return None
    checks = _mode_c_conditions(row, name, input_week_num)
    passed = len(select_mode_trades(pd.DataFrame([row]), name, mode["pattern"], mode["top_k"])) == 1
    if passed != all(check[-1] for check in checks):
        print("⚠ 條件明細與 TRADING_MODES 中 C 模式的設定不一致，結論以 TRADING_MODES 為準")
    conditions = pd.DataFrame(checks, columns=["條件", "數值", "C 模式要求", "符合"])
    conditions["符合"] = conditions["符合"].map({True: "✔", False: "✘"})
    monday = _week_monday(anchor)
    print(f"「{abbr}」錨點週 {anchor[0]}W{anchor[1]:02d} ({monday:%m/%d} ~ {monday + timedelta(days=6):%m/%d})，"
          f"錨點收盤 {row['錨點收盤']}")
    display(conditions)
    if not passed:
        failed = "、".join(conditions.loc[conditions["符合"] == "✘", "條件"])
        print(f"結論：不符合 C 模式 (沒通過：{failed})")
        return conditions, False, None
    plan = _recommendation_table(pd.DataFrame([row]), name, input_week_num, mode["buy_weeks"], today)
    print("結論：符合 C 模式，交易計畫如下 (欄位說明見「本週建議買入」)")
    display(plan)
    return conditions, True, plan

### 輸入公司代碼，檢查週資料(5份csv)
檢查 `DataCollection/代碼` 資料夾內的 5 份 csv，以下情況會用 DataScrape 重新抓取 (先抓新資料，全部成功後才刪除舊檔)：
* 沒有資料夾，或檔案有誤：缺檔、讀不到、欄位數不對、沒有資料
* 資料太舊：最新的完整週距離今天所在的週 >= 2 週 (例如今天在第 48 週，最新的完整週只到第 46 週)；這種情況抓取失敗時，會保留舊檔並繼續使用

In [8]:
# ===== 執行：輸入代碼 → 檢查 ABBR 資料夾的 5 份 CSV (沒有、有誤或太舊就用 DataScrape 重新取得) =====
# 重新取得時想看瀏覽器操作過程，可改用 ds.fetch_all_week_data(ABBR, headless=False)
ABBR = ds.ask_stock_code()
ABBR_DIR = ensure_week_data(ABBR)

請輸入台股代碼(4位數字)：2330
⚠ 資料夾「2330」的資料太舊：最新的完整週是 2026W46，今天在 2026W48，相差 2 週 (>= 2 週)
使用 WeekTermInvestment_DataScrape 抓取新資料，抓取成功後再刪除舊的 CSV 檔…
✔ 120 週的開高低收都與 K 線圖資料一致
已刪除舊檔：2330_Institutional_Trading_Week_8cols_20261118.csv
已刪除舊檔：2330_Major_Holders_Week_5cols_20261118.csv
已刪除舊檔：2330_Margin_Week_11cols_20261118.csv
已刪除舊檔：2330_Securities_Lending_and_Borrowing_Week_8cols_20261118.csv
已刪除舊檔：2330_Stock_Price_Week_20261118.csv
2330.TW Institutional_Trading_Week_8cols 共 52 筆，已存檔：C:\Users\Michael\workplace\python\WeekTermInvestment\DataCollection\2330\2330_Institutional_Trading_Week_8cols_20261126.csv
2330.TW Margin_Week_11cols 共 30 筆，已存檔：C:\Users\Michael\workplace\python\WeekTermInvestment\DataCollection\2330\2330_Margin_Week_11cols_20261126.csv
2330.TW Securities_Lending_and_Borrowing_Week_8cols 共 30 筆，已存檔：C:\Users\Michael\workplace\python\WeekTermInvestment\DataCollection\2330\2330_Securities_Lending_and_Borrowing_Week_8cols_20261126.csv
2330.TW Major_Holders_Week_5cols 共 50 筆，已存檔：C:\Users\Michae

### 分析參數
* `PIVOT_YEAR_WEEK`：選定的年與該年的第幾週，型別為 `(年, 第幾週)` 的 tuple，採 ISO 週次 (週一到週日為一週)，例如 `(2026, 45)` 是 2026/11/2 ~ 11/8 那一週；某天屬於第幾週可用 `date(2026, 11, 6).isocalendar()[:2]` 查。抓資料時還沒結束的週在讀取時會被排除，所以要選已經結束的週
* `INPUT_WEEK_NUM`：以 `PIVOT_YEAR_WEEK` 為終點(包含)，往前幾週的數據做數值計算：計算大戶與散戶的平均成本，並用 5 份 csv 中同一段期間的資料做出預測
* `OUTPUT_WEEK_NUM`：以 `PIVOT_YEAR_WEEK` 為起點(不包含)，往後幾週的股價，用來驗證是否照預測的方向變化；`PIVOT_YEAR_WEEK` 之後要有足夠的週資料才能驗證

In [10]:
PIVOT_YEAR_WEEK = (2026, 30)  # 選定的年與該年的第幾週 (ISO 年, 第幾週)
INPUT_WEEK_NUM = 12           # 以 PIVOT_YEAR_WEEK 為終點(包含)，往前幾週的數據做計算 (大戶/散戶平均成本等)
OUTPUT_WEEK_NUM = 4           # 以 PIVOT_YEAR_WEEK 為起點(不包含)，往後幾週的股價用來驗證預測的變化

### 大戶買入平均成本

In [ ]:
# ===== 執行：計算大戶買入平均成本 =====
avg_cost = calc_institutional_avg_cost(ABBR, PIVOT_YEAR_WEEK, INPUT_WEEK_NUM)
for name, cost in avg_cost.items():
    print(f"{ABBR} {name} 平均成本 ({PIVOT_YEAR_WEEK} 往前 {INPUT_WEEK_NUM} 週)：{cost:.2f}")

### 散戶買入平均成本

In [ ]:
# ===== 執行：計算散戶買入平均成本 =====
retail_cost = calc_retail_avg_cost(ABBR, PIVOT_YEAR_WEEK, INPUT_WEEK_NUM)
print(f"{ABBR} 散戶(融資) 平均成本 ({PIVOT_YEAR_WEEK} 往前 {INPUT_WEEK_NUM} 週)：{retail_cost:.2f}")

### 假設驗證(回測)：買進訊號 → 未來股價與大戶平均成本的關係
用 `DataCollection` 中所有 5 份 csv 齊全的股票，對每個 `INPUT_WEEK_NUM` (12 ~ 可用上限) 與每個可用的錨點回測，命中有兩種定義 (`HIT_RULES`)：
* `touch`：未來 `OUTPUT_WEEK_NUM` 週的最低價 <= 大戶成本 (有機會用 <= 成本的價格買進)
* `hold`：未來 `OUTPUT_WEEK_NUM` 週的最低價 >= 大戶成本 (股價守住大戶成本，成本是支撐)

看「同錨點同距離差」是否 > 0：在同一段時間、股價離成本差不多遠的情況下，有訊號的命中率比沒訊號的高，才代表訊號真的有用。

In [ ]:
# ===== 執行：假設驗證 (回測) =====
# 只讀取 DataCollection (DATA_DIR) 中現有的 csv，不會觸發 DataScrape；股票越多越久 (約 1900 檔時約需 10 分鐘以上)
BACKTEST_DATA_DIR = DATA_DIR
# 資券餘額、借券資料只有 30 週，還要留 OUTPUT_WEEK_NUM 週驗證，所以 INPUT_WEEK_NUM 目前最多約 25
BACKTEST_INPUT_WEEK_NUMS = range(12, 26)

backtest_results = backtest_buy_signal(BACKTEST_INPUT_WEEK_NUMS, OUTPUT_WEEK_NUM, BACKTEST_DATA_DIR)
# 回測結果與命中的定義無關，換定義只要重新彙整，不用重跑回測
for hit_rule, (description, _) in HIT_RULES.items():
    print(f"\n命中的定義 ({hit_rule})：{description}")
    display(summarize_backtest(backtest_results, hit_rule).round(3).T)

### 來回交易回測：提高來回成功率 (價差 >= 3%)
用 `DataCollection` 中 5 份 csv 齊全的股票 (不齊全的跳過，不會觸發 DataScrape)，以 `INPUT_WEEK_NUM` 週計算成本與特徵，買進、賣出各等最多 `OUTPUT_WEEK_NUM` 週，並使用 `ROUND_TRIP_STOP_LOSS` 停損 (也就是 A：4 週模式的交易規則)。這張表的報酬都還沒扣交易成本。

表格欄位：
* 機會數：符合條件的「股票 × 錨點週」數；平均每週機會數 = 機會數 / 錨點週數
* 價差中位數：散戶成本 / 大戶成本 - 1 的中位數 (兩邊都照成本價成交時的獲利)
* 買到率：錨點後 4 週內股價碰到大戶成本、買得到的比例
* 買到後成功率：買到之後，4 週內又碰到散戶成本、賣得到的比例
* 來回成功率：買到而且賣在散戶成本的比例 (= 買到率 × 買到後成功率)；前期 / 後期是錨點週分成前後兩半各自的結果，最差一週是單一錨點週的最低值
* 停損出場率：買到的交易中，因為週收盤跌破停損價而賣出的比例
* 成功時平均獲利：賣在散戶成本的交易，賣價 / 買價 - 1 的平均
* 未成功時平均報酬：買到但沒賣在散戶成本 (停損或到期賣出) 的交易的平均報酬
* 最差5%報酬：買到的交易中，報酬最差的 5% 那一筆的報酬
* 買到後平均報酬：所有買到的交易 (成功 + 未成功) 的平均報酬
* 成功平均持有週數：成功的交易從買進到賣出經過幾週

In [ ]:
# ===== 執行：來回交易回測 (買進 <= 大戶成本、賣出 >= 散戶成本) =====
# 只讀取 DataCollection (DATA_DIR) 中現有的 csv，不會觸發 DataScrape；約 1900 檔股票時約需 2 分鐘
# 樣本 (成本與特徵) 只算一次，未來股價保留到所有交易規則中最長的週數；下面的交易模式比較也沿用 round_trip_samples
future_weeks = [OUTPUT_WEEK_NUM * 2] + [m["buy_weeks"] + m["hold_weeks"] for m in TRADING_MODES.values()]
round_trip_samples = collect_round_trip_samples(INPUT_WEEK_NUM, max(future_weeks), min(future_weeks), DATA_DIR)
round_trip_results = apply_round_trip(round_trip_samples, OUTPUT_WEEK_NUM, OUTPUT_WEEK_NUM, ROUND_TRIP_STOP_LOSS)
for name in ROUND_TRIP_COSTS:
    print(f"\n大戶成本 = {name}")
    display(summarize_round_trip(round_trip_results, name).round(3))

### 交易模式比較 (大戶成本 = 外資)
沿用上一格的 `round_trip_samples`，比較 `TRADING_MODES` 的 A、B、C 三個模式 (規則見「函式定義(交易模式)」)：
* 全部：每個模式用它能回測的所有樣本。2 週模式只需要錨點後 4 週的股價，所以比 4 週模式多算最近幾個錨點週 (例如包含 2026 年 11 月的急跌)
* 同期：只看每個模式都能回測的樣本 (股票 × 錨點週)，用來公平比較
* C 只算「價差 / 週波動」<= `MODE_C_MAX_RATIO` (0.26) 的機會，同一週可能 0 檔，也可能好幾檔

表格欄位 (報酬都已扣掉交易成本 `ROUND_TRIP_TRADE_COST` = 0.585%：手續費 0.1425% × 2 + 證交稅 0.3%)：
* 錨點範圍、錨點週數：可以回測的錨點週；平均每週機會數 = 機會數 / 錨點週數
* 買到率、來回成功率、買到後成功率、停損出場率：定義同上一張表
* 平均淨報酬：買到的交易扣掉交易成本後的平均報酬
* 虧損率：買到的交易中，扣掉成本後虧錢的比例；虧損時平均：這些虧錢交易的平均報酬
* 最差一筆：買到的交易中淨報酬最差的一筆
* 賺錢週：把交易依錨點週分組，平均淨報酬 > 0 的週數 / 有買到的週數
* 平均資金占用週數：從買進那週到賣出那週共幾週 (包含買進那週)

In [ ]:
# ===== 執行：交易模式比較 (沿用上一格的 round_trip_samples，不用重新讀資料) =====
mode_table_all, mode_table_same_period = summarize_trading_modes(round_trip_samples, MODE_COST)
print("全部：每個模式能回測的所有樣本")
display(mode_table_all.round(3))
print("\n同期：只看每個模式都能回測的樣本")
display(mode_table_same_period.round(3))

### 本週建議買入 (C：精選 2 週模式，不停損)
用 `DataCollection` 現有的資料 (只讀取，不會修改 csv，也不會觸發 DataScrape)，每檔股票用「最新且 5 份資料都完整的週」當錨點：
* 抓資料時還沒結束的週 (檔名日期是週一 ~ 週五的那一週) 在讀取時就排除，所以錨點一定是完整的一週；各股票的錨點週可能不同
* 錨點比最新的錨點週早超過 1 週的股票 (例如這次沒更新到) 不列入
* 依 C 模式的條件篩選 (包含 價差 / 週波動 <= `MODE_C_MAX_RATIO`)，列出全部符合的股票；最新的錨點週在前，同一週依 價差 / 週波動 由小到大，資金不夠時先買前面的
* 只需要先執行上面的函式定義與「分析參數」(`INPUT_WEEK_NUM`)，不用先跑回測

表格欄位：
* 錨點週、錨點收盤：計算用的最新完整週，以及那一週的收盤價
* 大戶成本(買入價)：外資成本，買進期內以這個價格掛限價買單
* 散戶成本(賣出價)：買進的下一週起 2 週內，以這個價格掛限價賣單；沒賣到就在第 2 週週五收盤賣出
* 價差(%)：散戶成本 / 大戶成本 - 1
* 週波動(%)：錨點往前 12 週，每週 (最高 - 最低) / 收盤 的平均
* 價差/週波動：C 模式的條件是 <= 0.26，越小越優先
* 外資持股斜率(百分點/週)：大戶籌碼 csv 的外資持股比例，錨點往前 12 週的線性回歸斜率 (條件是 <= 0)
* 買進掛單價、賣出掛單價：依台股升降單位調整成可以掛單的價格 (買進往下取、賣出往上取)，與成本相差不到一個升降單位
* 買進期間、買進期狀態：錨點後 2 週的週一 ~ 週五，以及執行當天是否還在買進期內；「已結束」的就不要再買

In [13]:
# ===== 執行：本週建議買入 (C：精選 2 週模式，不停損) =====
# 只讀取 DataCollection (DATA_DIR) 中現有的 csv，不會觸發 DataScrape
recommendations, recommendation_candidates = recommend_trades("C：精選 2 週模式 (推薦)", INPUT_WEEK_NUM, DATA_DIR)
display(recommendations)
# recommendation_candidates 是同樣條件的全部候選；模式有 top_k (每週只買前幾檔) 時，才會比 recommendations 多

各錨點週可分析的股票：2026W47 360 檔、2026W46 1465 檔
略過：資料不齊全 113 檔、沒有可用的錨點 45 檔、資料太舊 2 檔；符合「精選：價差 <= 0.26 週波動、距大戶成本 -3% ~ +5% + 外資持股沒有增加」的候選 1 檔，建議買入 1 檔


,股票,錨點週,錨點收盤,大戶成本(買入價),散戶成本(賣出價),價差(%),週波動(%),價差/週波動,外資持股斜率(百分點/週),買進掛單價,賣出掛單價,買進期間,買進期狀態
0,1809,2026W46,45.1,43.15,44.53,3.22,13.44,0.239,-0.0276,43.1,44.55,11/16 ~ 11/27,進行中


### 個股 C 模式檢查
輸入股票代碼，用 `DataCollection` 現有的資料，判斷這檔股票最新的完整週是否符合 C 模式 (只讀取 csv，不會觸發 DataScrape)：
1. 前提檢查，不通過就提醒並結束：
   * 資料夾內 5 份 csv 都存在而且完整
   * 最新的完整週距離今天所在的週不能 >= 2 週 (例如今天在第 48 週，最新的完整週要是第 47 或 48 週)
   * 不通過時，先在「輸入公司代碼，檢查週資料」那一格輸入同一個代碼重新抓取，再回來執行這一格
2. 以最新的完整週當錨點 (通常是執行當週的前一週)，逐項檢查 C 模式的條件：大戶成本 < 散戶成本、價差 >= 3%、價差 / 週波動 <= `MODE_C_MAX_RATIO` (0.26)、錨點收盤距大戶成本 -3% ~ +5%、外資持股斜率 <= 0
3. 全部符合時，列出買進掛單價、賣出掛單價與買進期間 (欄位說明同「本週建議買入」)

只需要先執行上面的函式定義與「分析參數」(`INPUT_WEEK_NUM`)，不用先跑回測。

In [14]:
# ===== 執行：個股 C 模式檢查 =====
# 只讀取 DataCollection (DATA_DIR) 中現有的 csv，不會觸發 DataScrape；資料不完整或太舊時只會提醒並結束
CHECK_ABBR = ds.ask_stock_code()  # 只是請使用者輸入代碼，不會抓資料
mode_c_check = check_stock_mode_c(CHECK_ABBR, INPUT_WEEK_NUM, DATA_DIR)

請輸入台股代碼(4位數字)：1809
⚠ 「1809」最新的完整週是 2026W46，今天在 2026W48，相差 2 週 (>= 2 週)，資料太舊，請先在「輸入公司代碼，檢查週資料」那一格輸入同一個代碼，用 DataScrape 抓取最新資料
